# Tugas 1

In [ ]:
# Jika belum install library, jalankan sekali saja
!pip install requests beautifulsoup4 trafilatura pandas openpyxl tqdm


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import re
import time
import random
import requests
import pandas as pd
import trafilatura

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlsplit, urlunsplit
from tqdm import tqdm


# ============================================================
# 1. KONFIGURASI
# ============================================================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0 Safari/537.36"
    )
}

session = requests.Session()
session.headers.update(HEADERS)


kategori = {
    "sport": {
        "url": "https://sport.detik.com/indeks",
        "domain": "sport.detik.com"
    },

    "finance": {
        "url": "https://finance.detik.com/indeks",
        "domain": "finance.detik.com"
    }
}

In [ ]:
def normalize_url(url):
    """
    Menghapus query parameter dan fragment dari URL.
    """

    bagian = urlsplit(url)

    url_bersih = urlunsplit(
        (
            bagian.scheme,
            bagian.netloc,
            bagian.path,
            "",
            ""
        )
    )

    return url_bersih

In [ ]:
def ambil_link_berita(index_url, domain, target=150, max_page=20):

    links = []
    sudah_ada = set()

    for page in range(1, max_page + 1):

        url_page = f"{index_url}?page={page}"

        print(f"Membaca halaman {page}")

        try:

            response = session.get(
                url_page,
                timeout=20
            )

            response.raise_for_status()

        except Exception as e:

            print("Error:", e)
            continue


        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )


        for tag in soup.find_all("a", href=True):

            href = urljoin(
                url_page,
                tag["href"]
            )

            href = normalize_url(href)

            parsed = urlsplit(href)


            # URL artikel Detik biasanya memiliki /d-xxxx
            if (
                parsed.netloc == domain
                and re.search(r"/d-\d+", parsed.path)
            ):

                if href not in sudah_ada:

                    sudah_ada.add(href)

                    links.append(href)


        print(
            "Jumlah link:",
            len(links)
        )


        if len(links) >= target:
            break


        # delay
        time.sleep(
            random.uniform(1, 2)
        )


    return links

In [ ]:
link_sport = ambil_link_berita(
    kategori["sport"]["url"],
    kategori["sport"]["domain"],
    target=150
)

print(
    "Total kandidat Sport:",
    len(link_sport)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 40
Membaca halaman 3
Jumlah link: 60
Membaca halaman 4
Jumlah link: 80
Membaca halaman 5
Jumlah link: 98
Membaca halaman 6
Jumlah link: 118
Membaca halaman 7
Jumlah link: 137
Membaca halaman 8
Jumlah link: 157
Total kandidat Sport: 157


In [ ]:
link_finance = ambil_link_berita(
    kategori["finance"]["url"],
    kategori["finance"]["domain"],
    target=150
)

print(
    "Total kandidat Finance:",
    len(link_finance)
)

Membaca halaman 1
Jumlah link: 20
Membaca halaman 2
Jumlah link: 40
Membaca halaman 3
Jumlah link: 58
Membaca halaman 4
Jumlah link: 78
Membaca halaman 5
Jumlah link: 97
Membaca halaman 6
Jumlah link: 117
Membaca halaman 7
Jumlah link: 134
Membaca halaman 8
Jumlah link: 154
Total kandidat Finance: 154


In [ ]:
def bersihkan_teks(teks):

    if teks is None:
        return None


    # menghapus teks iklan
    teks = re.sub(
        r"ADVERTISEMENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    teks = re.sub(
        r"SCROLL TO CONTINUE WITH CONTENT",
        " ",
        teks,
        flags=re.IGNORECASE
    )


    # menghapus spasi berlebihan
    teks = re.sub(
        r"\s+",
        " ",
        teks
    )


    return teks.strip()

In [ ]:
def ambil_isi_berita(url):

    try:

        response = session.get(
            url,
            timeout=25
        )

        response.raise_for_status()


        isi = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False,
            output_format="txt",
            favor_precision=True,
            deduplicate=True
        )


        isi = bersihkan_teks(isi)


        # menghindari halaman kosong
        # atau isi yang terlalu pendek
        if isi and len(isi) >= 300:

            return isi


    except Exception as e:

        print(
            "Gagal:",
            url
        )

        print(
            "Error:",
            e
        )


    return None

In [ ]:
def scraping_kategori(
    links,
    label,
    target=100
):

    hasil = []


    for url in tqdm(
        links,
        desc=f"Scraping {label}"
    ):


        if len(hasil) >= target:
            break


        isi = ambil_isi_berita(url)


        if isi is not None:

            hasil.append(
                {
                    "isi_berita": isi,
                    "label": label
                }
            )


        # delay supaya tidak terlalu cepat
        time.sleep(
            random.uniform(1, 2)
        )


    print(
        f"{label} berhasil:",
        len(hasil)
    )


    return hasil

In [ ]:
data_sport = scraping_kategori(
    link_sport,
    "sport",
    target=100
)

Scraping sport:  65%|██████▍   | 102/157 [03:51<02:04,  2.27s/it]

sport berhasil: 100


In [ ]:
data_finance = scraping_kategori(
    link_finance,
    "finance",
    target=100
)

Scraping finance:  66%|██████▌   | 102/154 [04:02<02:03,  2.37s/it]

finance berhasil: 100


In [ ]:
data_semua = (
    data_sport
    +
    data_finance
)

In [ ]:
df = pd.DataFrame(
    data_semua
)

In [ ]:
df.insert(
    0,
    "id",
    range(
        1,
        len(df) + 1
    )
)

In [ ]:
df = df[
    [
        "id",
        "isi_berita",
        "label"
    ]
]

In [ ]:
df

,id,isi_berita,label
0,1,Chef de Mission (CdM) Indonesia Todotua Pasari...,sport
1,2,"Banjir besar melanda Nagoya, Jepang, menjelang...",sport
2,3,Ketua Umum Komite Olimpiade Indonesia (KOI) Ra...,sport
3,4,Persaingan titel juara dunia semakin ketat men...,sport
4,5,Sektor ganda campuran kembali mengalami peromb...,sport
...,...,...,...
195,196,Putri Mahkota Victoria Ingrid Alice Desiree be...,finance
196,197,Pemerintah terus menggenjot pengembangan energ...,finance
197,198,Lembaga Pembiayaan Ekspor Indonesia (LPEI)/Ind...,finance
198,199,Menteri Koordinator Bidang Perekonomian Airlan...,finance


In [ ]:
print(
    "Total data:",
    len(df)
)

Total data: 200


In [ ]:
print(
    df["label"].value_counts()
)

label
sport      100
finance    100
Name: count, dtype: int64


In [ ]:
nama_file = "dataset_detik_200_berita.xlsx"


df.to_excel(
    nama_file,
    index=False
)


print(
    "File berhasil disimpan:",
    nama_file
)

File berhasil disimpan: dataset_detik_200_berita.xlsx


In [ ]:
import os

print(os.path.abspath("dataset_detik_200_berita.xlsx"))

D:\PPW\enWebmining\Anjar\dataset_detik_200_berita.xlsx
